# LLM Serving Systems in Practice

This notebook ties together everything from the series — the optimisations we've covered
are not used in isolation. They're composed into **serving systems** that handle real
production traffic.

We'll compare:
1. The serving stack — what a request goes through
2. vLLM — PagedAttention and continuous batching
3. SGLang — RadixAttention and compressed FSMs
4. TensorRT-LLM — NVIDIA's compiler-optimised runtime
5. llama.cpp — CPU/edge inference
6. Choosing the right system
7. Key metrics and how to benchmark

## 1. The Serving Stack — Anatomy of a Request

When a user sends a prompt to an LLM API, it passes through multiple layers:

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Client request: "Explain quantum computing"                            │
└───────────────────────────────┬─────────────────────────────────────────┘
                                ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  API Gateway / Load Balancer                                            │
│  • Rate limiting, auth, routing                                         │
│  • Routes to least-loaded replica                                       │
└───────────────────────────────┬─────────────────────────────────────────┘
                                ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  Serving Engine (vLLM / SGLang / TRT-LLM)                              │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Scheduler                                                       │   │
│  │  • Request queue management                                      │   │
│  │  • Continuous batching decisions                                  │   │
│  │  • Preemption / priority                                         │   │
│  │  • Memory (KV cache) allocation                                  │   │
│  └──────────────────────┬──────────────────────────────────────────┘   │
│                         ▼                                               │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Execution Engine                                                │   │
│  │  • Tokenisation                                                  │   │
│  │  • Prefill (batch prompt processing)                             │   │
│  │  • Decode loop (token-by-token with KV cache)                    │   │
│  │  • Sampling (temperature, top-p, top-k)                          │   │
│  │  • Detokenisation + streaming                                    │   │
│  └──────────────────────┬──────────────────────────────────────────┘   │
│                         ▼                                               │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Model Backend                                                   │   │
│  │  • CUDA kernels (FlashAttention, quantised matmul)               │   │
│  │  • Tensor parallelism communication                              │   │
│  │  • KV cache management (paged / radix)                           │   │
│  └─────────────────────────────────────────────────────────────────┘   │
└───────────────────────────────┬─────────────────────────────────────────┘
                                ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  Streaming response: "Quantum computing is..."                          │
└─────────────────────────────────────────────────────────────────────────┘
```

The serving engine is where all the optimisations from our series live. Different
systems make different choices at each layer.

## 2. vLLM — The Production Standard

**vLLM** (UC Berkeley, 2023) introduced **PagedAttention** and popularised continuous
batching for LLM serving. It's the most widely deployed open-source serving engine.

### Key innovations

#### PagedAttention
Traditional KV cache allocates a contiguous block for each request's maximum possible
length. This wastes memory on requests that generate fewer tokens.

PagedAttention borrows from OS virtual memory:

```
Traditional (contiguous):          PagedAttention (paged):

Request A: [████████░░░░░░░░]     Request A: [page1][page2][page3]
           allocated but unused              only allocate as needed
Request B: [██████████░░░░░░]     Request B: [page1][page2][page3][page4]
           fragmented gaps          
                                   Free pages: [░][░][░][░][░][░]...
   30-50% memory wasted             <5% waste (page granularity only)
```

**Result**: ~2-4x more concurrent requests fit in the same GPU memory.

#### Other features
- Continuous batching (iteration-level scheduling)
- Prefix caching (automatic, hash-based)
- Speculative decoding (draft model or Medusa)
- Tensor parallelism + pipeline parallelism
- FP8/INT4 quantisation (AWQ, GPTQ, SqueezeLLM)
- Chunked prefill
- LoRA serving (multiple adapters)

### When to use vLLM
- Default choice for most production deployments
- Large model serving (TP/PP support)
- High-throughput APIs
- OpenAI-compatible API endpoint

## 3. SGLang — Structured Generation and Prefix Sharing

**SGLang** (LMSYS, 2024) focuses on **programmatic LLM usage** — where the same model
is called repeatedly with shared prefixes, structured outputs, and multi-turn patterns.

### Key innovations

#### RadixAttention (prefix caching via radix tree)

```
Traditional: each request computes its own KV cache from scratch.
SGLang: a radix tree stores KV cache by token prefix.
        Requests sharing prefixes reuse cached KV entries.

                   [system prompt: 800 tokens]
                  /                           \
         [few-shot A: 200 tok]          [few-shot B: 200 tok]
         /          \                         |
   [query 1]    [query 2]               [query 3]
   
Query 1: skips prefilling 1000 tokens (system + few-shot A already cached)
Query 3: skips prefilling 1000 tokens (system + few-shot B already cached)
```

Up to **6.4x throughput** improvement for workloads with shared prefixes.

#### Compressed Finite State Machines
For structured output (JSON, regex constraints), SGLang pre-compiles the grammar
into a compressed FSM and applies it during decoding with minimal overhead.

#### Overlap scheduling
Overlaps CPU work (tokenisation, sampling, FSM transitions) with GPU compute
so neither stalls the other.

### When to use SGLang
- Heavy prefix sharing (chat with system prompts, RAG)
- Structured output (JSON mode, constrained generation)
- Multi-call LLM programs (chains, agents)
- Workloads where prefix caching matters more than raw single-request latency

## 4. TensorRT-LLM — NVIDIA's Compiled Runtime

**TensorRT-LLM** (NVIDIA) takes a different approach: **compile** the model into
highly optimised CUDA kernels ahead of time, with hardware-specific fusion.

### Key innovations

#### Kernel fusion
Instead of launching separate kernels for LayerNorm → Linear → GeLU → Linear,
TRT-LLM fuses them into a single kernel that keeps data in registers/SRAM:

```
PyTorch (eager):                    TRT-LLM (fused):

kernel_launch(LayerNorm)            kernel_launch(Fused_LN_Linear_GeLU_Linear)
  → write to HBM                     → everything stays in registers
kernel_launch(Linear)                 → one HBM round-trip total
  → write to HBM                     → 4x fewer kernel launches
kernel_launch(GeLU)
  → write to HBM
kernel_launch(Linear)
  → write to HBM

4 HBM round-trips                  1 HBM round-trip
4 kernel launches (~28us)          1 kernel launch (~7us)
```

#### FP8 native support
Leverages H100's FP8 tensor cores for 2x throughput over FP16 with negligible quality loss.

#### In-flight batching
NVIDIA's term for continuous batching with additional optimisations around
batch composition and memory management.

### Tradeoffs
- Highest raw performance (5-20% faster than vLLM on same hardware)
- But: longer setup time (compilation), less flexible, NVIDIA-only
- Model support can lag behind (new architectures need manual integration)

### When to use TRT-LLM
- Maximum throughput on NVIDIA hardware
- Latency-critical production (every millisecond counts)
- Stable model (not changing frequently)
- Have engineering capacity for the setup complexity

## 5. llama.cpp — CPU and Edge Inference

**llama.cpp** (Georgi Gerganov, 2023) makes LLMs run efficiently on consumer
hardware — CPUs, Apple Silicon, and small GPUs — via extreme quantisation.

### Key features

- **GGUF format**: flexible mixed-precision quantisation (2-8 bit per layer)
- **CPU-optimised**: AVX2/AVX-512 SIMD, ARM NEON for Apple Silicon
- **Metal/CUDA offload**: optionally offload layers to GPU
- **No Python overhead**: pure C/C++ inference
- **Memory-mapped weights**: stream from disk, don't need all in RAM

### Quantisation tiers

```
GGUF quantisation levels:
  Q2_K:  ~2.5 bits/weight — barely usable, massive quality loss
  Q3_K_M: ~3.4 bits/weight — noticeable quality loss
  Q4_K_M: ~4.5 bits/weight — good balance (most popular)
  Q5_K_M: ~5.5 bits/weight — near-lossless
  Q6_K:  ~6.5 bits/weight — indistinguishable from FP16
  Q8_0:  ~8.5 bits/weight — reference quality
  
  70B model sizes:
  Q4_K_M: ~40 GB (fits in 64 GB RAM Mac)
  Q6_K:   ~57 GB
  Q8_0:   ~75 GB
```

### The Apple Silicon advantage

Apple's unified memory architecture means the GPU and CPU share the same RAM:
- M2 Ultra: 192 GB unified memory, 800 GB/s bandwidth
- No PCIe transfer needed — GPU kernels access model weights directly
- 70B Q4 at ~20-30 tok/s on a Mac Studio

### When to use llama.cpp
- Local/offline inference (privacy, no API costs)
- Edge deployment (phones, laptops, IoT)
- CPU-only servers
- Rapid prototyping and experimentation
- When you don't have datacenter GPUs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Feature comparison matrix
systems = ['vLLM', 'SGLang', 'TRT-LLM', 'llama.cpp']
features = [
    'Continuous batching',
    'PagedAttention',
    'Prefix caching',
    'Speculative decoding',
    'Structured output',
    'Tensor parallelism',
    'FP8 support',
    'INT4/GPTQ/AWQ',
    'CPU inference',
    'Apple Silicon',
    'Kernel fusion',
    'Multi-LoRA',
]

# Support matrix (0=no, 1=partial, 2=full/best-in-class)
support = np.array([
    # vLLM  SGLang  TRT   llama.cpp
    [2,     2,      2,    0],  # Continuous batching
    [2,     1,      1,    0],  # PagedAttention
    [2,     2,      1,    0],  # Prefix caching
    [2,     2,      2,    1],  # Speculative decoding
    [1,     2,      1,    1],  # Structured output
    [2,     2,      2,    0],  # Tensor parallelism
    [2,     2,      2,    0],  # FP8
    [2,     2,      2,    2],  # INT4
    [0,     0,      0,    2],  # CPU inference
    [0,     0,      0,    2],  # Apple Silicon
    [1,     1,      2,    1],  # Kernel fusion
    [2,     1,      1,    1],  # Multi-LoRA
])

fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.RdYlGn
im = ax.imshow(support, cmap=cmap, aspect='auto', vmin=0, vmax=2)

ax.set_xticks(range(len(systems)))
ax.set_xticklabels(systems, fontsize=12, fontweight='bold')
ax.set_yticks(range(len(features)))
ax.set_yticklabels(features, fontsize=10)
ax.set_title('Serving System Feature Comparison', fontsize=14)

# Add text annotations
labels = {0: '✗', 1: '◐', 2: '✓'}
for i in range(len(features)):
    for j in range(len(systems)):
        ax.text(j, i, labels[support[i, j]], ha='center', va='center', fontsize=14)

plt.colorbar(im, ax=ax, ticks=[0, 1, 2], label='Support level')
plt.tight_layout()
plt.show()

print("✓ = full/best-in-class  ◐ = partial/basic  ✗ = not supported")

## 6. Choosing the Right System

### Decision tree

```
What hardware are you targeting?
├── Consumer hardware / CPU / Mac
│   └── llama.cpp (GGUF quantisation)
│
├── NVIDIA datacenter GPUs (A100/H100)
│   ├── Need maximum raw throughput + stable model?
│   │   └── TensorRT-LLM
│   │
│   ├── Heavy prefix sharing / structured output / multi-call patterns?
│   │   └── SGLang
│   │
│   └── General production serving / OpenAI-compatible API?
│       └── vLLM (safest default)
│
└── AMD GPUs (MI300X)
    └── vLLM (ROCm support) or SGLang
```

### Typical production architectures

| Workload | System | Config | Why |
|----------|--------|--------|-----|
| Chat API (GPT-4 competitor) | vLLM | TP=8, chunked prefill | General purpose, proven |
| RAG with shared docs | SGLang | Prefix caching ON | 6x throughput from prefix reuse |
| Code completion | vLLM + speculative | Draft model | High acceptance rate on code |
| JSON extraction at scale | SGLang | Compressed FSMs | Guaranteed valid JSON |
| Edge / on-device | llama.cpp | Q4_K_M | Fits in consumer RAM |
| Latency-critical trading | TRT-LLM | FP8, fused kernels | Every ms matters |
| Multi-tenant (many LoRAs) | vLLM | S-LoRA | Efficient adapter serving |

## 7. Key Metrics and How to Benchmark

### The four numbers that matter

| Metric | What it measures | Who cares |
|--------|-----------------|----------|
| **TTFT** (Time to First Token) | Latency before output starts | User experience (streaming) |
| **ITL** (Inter-Token Latency) | Gap between consecutive tokens | Streaming smoothness |
| **Throughput** (tok/s) | Total output tokens per second | Cost/capacity planning |
| **p99 latency** | Worst-case experience | SLA compliance |

### Tradeoffs between metrics

```
                    Higher batch size
                    ─────────────────►
        ┌──────────────────────────────────────┐
        │                                      │
  Low   │  ★ Best latency      ● Best         │  High
  batch │   (batch=1)           throughput     │  batch
        │   Low throughput      Higher latency │
        │                                      │
        └──────────────────────────────────────┘
        
You can't optimise for both simultaneously.
Serving systems let you pick your tradeoff point.
```

### Benchmarking tools

| Tool | What it does |
|------|-------------|
| `vllm benchmark` | Built-in load testing for vLLM |
| `genai-perf` (NVIDIA) | Comprehensive LLM benchmarking |
| `llmperf` (Anyscale) | Cross-system comparison |
| ShareGPT traces | Realistic multi-turn conversation workloads |

### What to measure

```python
# Key parameters to sweep:
# 1. Request rate (requests/sec) — from light to saturating
# 2. Input length distribution (short prompts vs long RAG contexts)
# 3. Output length distribution (short answers vs long generation)
# 4. Concurrency (how many simultaneous requests)

# Report:
# - TTFT p50, p90, p99
# - ITL p50, p90, p99
# - Throughput at each concurrency level
# - Maximum sustainable request rate under SLA
```

In [ ]:
# Simulate throughput vs latency tradeoff at different batch sizes

def simulate_serving_metrics(batch_sizes, model_gb=14, gpu_bw=3.35, overhead_ms=0.5):
    """Simplified model of throughput vs latency at different batch sizes."""
    results = []
    for bs in batch_sizes:
        # Weight load time (constant regardless of batch)
        weight_time = model_gb * 1e9 / (gpu_bw * 1e12) * 1000  # ms
        # Compute time scales with batch (matters when compute-bound)
        compute_time = bs * 0.01  # ~0.01ms per item at small batch
        # Latency = max(memory_time, compute_time) + overhead
        latency = max(weight_time, compute_time) + overhead_ms
        # Throughput = batch_size / latency
        throughput = bs / latency * 1000  # tok/s
        results.append({"batch": bs, "latency_ms": latency, "throughput": throughput})
    return results

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256]
results = simulate_serving_metrics(batch_sizes)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

latencies = [r['latency_ms'] for r in results]
throughputs = [r['throughput'] for r in results]

ax1.plot(batch_sizes, latencies, 'r-o', linewidth=2)
ax1.set_xlabel('Batch size')
ax1.set_ylabel('Latency per token (ms)')
ax1.set_title('Latency increases with batch')
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log', base=2)

ax2.plot(batch_sizes, throughputs, 'g-o', linewidth=2)
ax2.set_xlabel('Batch size')
ax2.set_ylabel('Throughput (tok/s)')
ax2.set_title('Throughput scales with batch (until saturation)')
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log', base=2)

# The tradeoff frontier
ax3.scatter(throughputs, latencies, c=batch_sizes, cmap='viridis', s=100, zorder=5)
ax3.plot(throughputs, latencies, 'k--', alpha=0.3)
for r in results[::2]:
    ax3.annotate(f'b={r["batch"]}', (r['throughput'], r['latency_ms']),
                 textcoords="offset points", xytext=(5, 5), fontsize=8)
ax3.set_xlabel('Throughput (tok/s)')
ax3.set_ylabel('Latency (ms)')
ax3.set_title('The throughput-latency tradeoff')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Batch=1:   {results[0]['latency_ms']:.1f} ms latency, {results[0]['throughput']:.0f} tok/s")
print(f"Batch=32:  {results[5]['latency_ms']:.1f} ms latency, {results[5]['throughput']:.0f} tok/s")
print(f"Batch=256: {results[8]['latency_ms']:.1f} ms latency, {results[8]['throughput']:.0f} tok/s")
print(f"\nBatch 1→32: latency +{results[5]['latency_ms']/results[0]['latency_ms']:.1f}x, throughput +{results[5]['throughput']/results[0]['throughput']:.0f}x")
print(f"Continuous batching dynamically adjusts batch size to balance these.")

## Summary — The Optimisation Stack

Everything in this notebook series composes into the serving systems we've compared:

```
Notebook  Concept                      Where it appears in serving
────────  ───────────────────────────  ──────────────────────────────────────
01        How LLMs work                The model being served
02        KV cache                     Core of every serving engine
03/03a    GPU fundamentals + memory    Why decode is memory-bound
04        Batching strategies          Scheduler (continuous batching)
05        Speculative decoding         Decode optimisation in vLLM/SGLang
06        Quantisation                 Weight format (INT4/FP8)
07        Attention variants           GQA reduces KV cache, FlashAttention
08        Model parallelism            Multi-GPU serving configuration
09        Serving systems              This notebook — putting it all together
```

### The key principles (from most to least impactful)

1. **Reduce bytes loaded per token** — quantisation (3-4x speedup)
2. **Batch to amortise weight loads** — continuous batching (10-30x throughput)
3. **Don't recompute** — KV cache + prefix caching (2-6x)
4. **Generate multiple tokens per weight load** — speculative decoding (2-3x latency)
5. **Keep data close to compute** — FlashAttention, kernel fusion (2-4x for attention)
6. **Scale bandwidth with more GPUs** — tensor parallelism (near-linear)
7. **Use the right model size** — cascade routing (2x cost reduction)